# 面试题：RAG 里的关键词检索应该怎样设计？

## 可以直接复述的回答

RAG 的关键词检索不是“搜到一篇文档”就结束，而是查询理解、多个召回路由、元数据门禁、候选融合、上下文装配和引用生成组成的流水线。原始问题与同义词改写应该分别召回，随后用 rank-based 融合减少不同通道分数量纲的影响。ACL、有效版本和租户过滤必须在候选进入上下文前执行，不能寄希望于模型自行忽略敏感内容。答案中的引用要来自实际进入上下文的 chunk，并保存 source、section 和版本，才能审计。本题用 8 个客服知识 chunk 与 6 个口语问题，手写关键词加权召回、同义词扩展、RRF 融合和带预算的引用答案。最后复现“内部稿被晚过滤而泄漏进上下文”的失败案例。

## 真实案例

数据模拟帮助中心 chunk，字段包含 chunk_id、标题、tokens、正文、来源、章节、可见性和有效版本。它是无需联网的脱敏教学集，答案采用抽取式模板，不代表大模型生成质量。

In [1]:
import math  # 导入对数函数以计算关键词稀有度
from collections import Counter  # 导入计数器以统计 chunk 文档频率
from pprint import pprint  # 导入结构化打印函数以展示候选和引用
chunks = [{"id": "R_old", "标题": "旧版退款到账", "tokens": ["退钱", "多久", "退款", "到账", "七天"], "正文": "旧流程写的是七天到账，已经停用。", "source": "help/refund-v1.md", "section": "到账", "visibility": "public", "active": False}, {"id": "R_now", "标题": "退款到账时限", "tokens": ["退款", "到账", "时限", "原路", "工作日"], "正文": "退款审核通过后，通常三个工作日原路到账。", "source": "help/refund.md", "section": "到账时限", "visibility": "public", "active": True}, {"id": "I_internal", "标题": "发票内部审批稿", "tokens": ["抬头", "税号", "发票", "公司", "名称", "纳税号", "内部"], "正文": "内部审批账号与操作口令仅供员工使用。", "source": "internal/invoice.md", "section": "审批", "visibility": "internal", "active": True}, {"id": "I_public", "标题": "电子发票抬头修改", "tokens": ["电子", "发票", "公司", "名称", "纳税号", "修改"], "正文": "在电子发票页修改公司名称和纳税号后重新开具。", "source": "help/invoice.md", "section": "抬头修改", "visibility": "public", "active": True}, {"id": "A_reset", "标题": "账号密码重置", "tokens": ["账号", "密码", "重置", "登录", "验证码"], "正文": "无法登录时先验证手机号，再重置密码。", "source": "help/account.md", "section": "密码重置", "visibility": "public", "active": True}, {"id": "O_cancel", "标题": "取消未发货订单", "tokens": ["订单", "取消", "未发货", "关闭", "支付"], "正文": "未发货订单可在订单详情选择取消。", "source": "help/order.md", "section": "取消订单", "visibility": "public", "active": True}, {"id": "R_proof", "标题": "退款凭证上传", "tokens": ["退款", "凭证", "照片", "破损", "上传"], "正文": "商品破损退款需要上传外包装和商品照片。", "source": "help/refund.md", "section": "凭证要求", "visibility": "public", "active": True}, {"id": "L_address", "标题": "修改收货地址", "tokens": ["收货", "地址", "修改", "发货", "物流"], "正文": "发货前可在订单详情修改收货地址。", "source": "help/logistics.md", "section": "地址修改", "visibility": "public", "active": True}]  # 构造八个含版本与权限字段的知识 chunk
questions = [{"id": "Q1", "问题": "退钱多久", "tokens": ["退钱", "多久"], "expected": "R_now"}, {"id": "Q2", "问题": "抬头和税号怎么改", "tokens": ["抬头", "税号"], "expected": "I_public"}, {"id": "Q3", "问题": "登录不了怎么处理", "tokens": ["登录不了"], "expected": "A_reset"}, {"id": "Q4", "问题": "未发货怎么撤单", "tokens": ["撤单", "未发货"], "expected": "O_cancel"}, {"id": "Q5", "问题": "破损退款要什么证明", "tokens": ["破损", "证明"], "expected": "R_proof"}, {"id": "Q6", "问题": "收货地还能改吗", "tokens": ["收货地"], "expected": "L_address"}]  # 构造六条口语问题及人工期望引用
print("RAG chunk 输入预览：")  # 输出案例标题
pprint([{key: chunk[key] for key in ["id", "标题", "source", "section", "visibility", "active"]} for chunk in chunks])  # 展示检索和审计所需字段
print("用户问题与期望 chunk：")  # 输出评估问题标题
pprint(questions)  # 展示六条问题及其人工标签

RAG chunk 输入预览：
[{'active': False,
  'id': 'R_old',
  'section': '到账',
  'source': 'help/refund-v1.md',
  'visibility': 'public',
  '标题': '旧版退款到账'},
 {'active': True,
  'id': 'R_now',
  'section': '到账时限',
  'source': 'help/refund.md',
  'visibility': 'public',
  '标题': '退款到账时限'},
 {'active': True,
  'id': 'I_internal',
  'section': '审批',
  'source': 'internal/invoice.md',
  'visibility': 'internal',
  '标题': '发票内部审批稿'},
 {'active': True,
  'id': 'I_public',
  'section': '抬头修改',
  'source': 'help/invoice.md',
  'visibility': 'public',
  '标题': '电子发票抬头修改'},
 {'active': True,
  'id': 'A_reset',
  'section': '密码重置',
  'source': 'help/account.md',
  'visibility': 'public',
  '标题': '账号密码重置'},
 {'active': True,
  'id': 'O_cancel',
  'section': '取消订单',
  'source': 'help/order.md',
  'visibility': 'public',
  '标题': '取消未发货订单'},
 {'active': True,
  'id': 'R_proof',
  'section': '凭证要求',
  'source': 'help/refund.md',
  'visibility': 'public',
  '标题': '退款凭证上传'},
 {'active': True,
  'id': 'L_address',
 

## Baseline / 基线：原词重叠且不做门禁

这个基线只计算用户原词与 chunk token 的重叠，并允许旧版本与内部文档参与排名。它会命中口语完全一致但已经停用的退款文档，也可能把内部发票稿送入上下文。

In [2]:
def overlap_score(terms, chunk):  # 定义原始词重叠基线分数
    return sum(term in chunk["tokens"] for term in dict.fromkeys(terms))  # 统计去重 query term 的命中数
def baseline_rank(question):  # 定义不做改写和权限过滤的基线排序
    rows = [{"chunk_id": chunk["id"], "score": overlap_score(question["tokens"], chunk)} for chunk in chunks]  # 计算所有 chunk 的原词重叠分
    return sorted(rows, key=lambda row: (-row["score"], row["chunk_id"]))  # 按重叠分和编号稳定排序
baseline_rankings = {question["id"]: baseline_rank(question) for question in questions}  # 运行六条问题的基线召回
baseline_hits = sum(baseline_rankings[question["id"]][0]["chunk_id"] == question["expected"] for question in questions)  # 统计基线 Top1 命中
print("不带改写与门禁的 Baseline Top3：")  # 输出基线结果标题
pprint([{"问题": question["问题"], "期望": question["expected"], "Top3": baseline_rankings[question["id"]][:3]} for question in questions])  # 展示逐问题基线结果
print(f"Baseline Top1 命中：{baseline_hits}/{len(questions)}")  # 输出基线汇总指标

不带改写与门禁的 Baseline Top3：
[{'Top3': [{'chunk_id': 'R_old', 'score': 2},
           {'chunk_id': 'A_reset', 'score': 0},
           {'chunk_id': 'I_internal', 'score': 0}],
  '期望': 'R_now',
  '问题': '退钱多久'},
 {'Top3': [{'chunk_id': 'I_internal', 'score': 2},
           {'chunk_id': 'A_reset', 'score': 0},
           {'chunk_id': 'I_public', 'score': 0}],
  '期望': 'I_public',
  '问题': '抬头和税号怎么改'},
 {'Top3': [{'chunk_id': 'A_reset', 'score': 0},
           {'chunk_id': 'I_internal', 'score': 0},
           {'chunk_id': 'I_public', 'score': 0}],
  '期望': 'A_reset',
  '问题': '登录不了怎么处理'},
 {'Top3': [{'chunk_id': 'O_cancel', 'score': 1},
           {'chunk_id': 'A_reset', 'score': 0},
           {'chunk_id': 'I_internal', 'score': 0}],
  '期望': 'O_cancel',
  '问题': '未发货怎么撤单'},
 {'Top3': [{'chunk_id': 'R_proof', 'score': 1},
           {'chunk_id': 'A_reset', 'score': 0},
           {'chunk_id': 'I_internal', 'score': 0}],
  '期望': 'R_proof',
  '问题': '破损退款要什么证明'},
 {'Top3': [{'chunk_id': 'A_reset', 'sco

## 手写召回与融合：别名扩展、稀有词加权、前置门禁、RRF

别名表把口语表达展开为知识库词汇，但保留原词路由以避免改写丢信息。两个路由分别排序后用 RRF 合并；有效版本和 visibility 在打分前过滤，内部内容不会进入候选池。

In [3]:
alias_map = {"退钱": ["退款", "到账"], "多久": ["时限"], "抬头": ["发票", "公司", "名称"], "税号": ["纳税号"], "登录不了": ["登录", "密码", "重置"], "撤单": ["订单", "取消"], "证明": ["凭证", "照片"], "收货地": ["收货", "地址", "修改"]}  # 定义可审计的小型领域别名表
document_frequency = Counter()  # 创建 chunk 级文档频率计数器
for chunk in chunks:  # 遍历全部 chunk 统计语料词频
    for token in set(chunk["tokens"]):  # 每个词在单个 chunk 只计一次
        document_frequency[token] += 1  # 累加包含该词的 chunk 数
def expand_terms(raw_terms):  # 定义保留原词的确定性查询扩展
    expanded = list(raw_terms)  # 先复制用户原始词
    for term in raw_terms:  # 逐个查看是否存在领域别名
        expanded.extend(alias_map.get(term, []))  # 追加当前词对应的规范词
    return list(dict.fromkeys(expanded))  # 去重并保持可解释顺序
def weighted_rank(terms, audience="public", enforce_gate=True, limit=4):  # 定义带元数据门禁的关键词召回
    rows = []  # 收集满足条件的候选及分数
    for chunk in chunks:  # 遍历知识库 chunk
        allowed = chunk["active"] and chunk["visibility"] == audience  # 判断版本和可见性是否允许进入上下文
        if enforce_gate and not allowed:  # 门禁开启时提前丢弃禁止内容
            continue  # 跳过无权或失效的 chunk
        score = 0.0  # 初始化稀有词加权分数
        matched = []  # 记录命中的 query term 便于解释
        for term in dict.fromkeys(terms):  # 遍历去重后的查询词
            if term in chunk["tokens"]:  # 仅对实际命中的词加分
                score += 1 + math.log((len(chunks) + 1) / (document_frequency[term] + 1))  # 用平滑 IDF 奖励稀有词
                matched.append(term)  # 保存命中词供候选账本展示
        if score > 0:  # 只保留至少命中一个关键词的候选
            rows.append({"chunk_id": chunk["id"], "score": round(score, 4), "matched": matched})  # 保存候选分数和命中证据
    return sorted(rows, key=lambda row: (-row["score"], row["chunk_id"]))[:limit]  # 返回稳定排序后的前若干候选
print("查询改写示例：")  # 输出查询理解标题
pprint([{"原词": question["tokens"], "扩展后": expand_terms(question["tokens"])} for question in questions])  # 展示六条问题的确定性扩展

查询改写示例：
[{'原词': ['退钱', '多久'], '扩展后': ['退钱', '多久', '退款', '到账', '时限']},
 {'原词': ['抬头', '税号'], '扩展后': ['抬头', '税号', '发票', '公司', '名称', '纳税号']},
 {'原词': ['登录不了'], '扩展后': ['登录不了', '登录', '密码', '重置']},
 {'原词': ['撤单', '未发货'], '扩展后': ['撤单', '未发货', '订单', '取消']},
 {'原词': ['破损', '证明'], '扩展后': ['破损', '证明', '凭证', '照片']},
 {'原词': ['收货地'], '扩展后': ['收货地', '收货', '地址', '修改']}]


In [4]:
def reciprocal_rank_fusion(route_rankings, constant=20):  # 定义不依赖原始分数量纲的 RRF 融合
    fused = {}  # 创建按 chunk 聚合的融合账本
    for route_name, ranking in route_rankings.items():  # 遍历原词与扩展词两个召回路由
        for rank, row in enumerate(ranking, start=1):  # 遍历当前路由的候选名次
            entry = fused.setdefault(row["chunk_id"], {"chunk_id": row["chunk_id"], "rrf": 0.0, "路由名次": {}, "命中词": {}})  # 初始化或读取候选融合记录
            entry["rrf"] += 1 / (constant + rank)  # 按名次累加 RRF 分数
            entry["路由名次"][route_name] = rank  # 记录候选来自哪个路由及名次
            entry["命中词"][route_name] = row["matched"]  # 保存该路由的关键词证据
    return sorted(fused.values(), key=lambda row: (-row["rrf"], row["chunk_id"]))  # 返回融合后的稳定排名
def retrieve_question(question, audience="public", enforce_gate=True):  # 定义一条问题的完整关键词召回流水线
    original_route = weighted_rank(question["tokens"], audience, enforce_gate)  # 用用户原词执行第一路召回
    expanded_route = weighted_rank(expand_terms(question["tokens"]), audience, enforce_gate)  # 用规范化扩展词执行第二路召回
    fused = reciprocal_rank_fusion({"原词": original_route, "扩展词": expanded_route})  # 融合两个候选列表
    return {"原词路由": original_route, "扩展词路由": expanded_route, "融合": fused}  # 返回完整候选账本
retrieval_runs = {question["id"]: retrieve_question(question) for question in questions}  # 执行六条公开用户问题
print("Q2 候选融合账本（内部稿应完全缺席）：")  # 输出关键中间量标题
pprint(retrieval_runs["Q2"])  # 展示原词、扩展词和 RRF 的候选来源

Q2 候选融合账本（内部稿应完全缺席）：
{'原词路由': [],
 '扩展词路由': [{'chunk_id': 'I_public',
            'matched': ['发票', '公司', '名称', '纳税号'],
            'score': 8.3944}],
 '融合': [{'chunk_id': 'I_public',
         'rrf': 0.047619047619047616,
         '命中词': {'扩展词': ['发票', '公司', '名称', '纳税号']},
         '路由名次': {'扩展词': 1}}]}


## 上下文装配、引用与结果解读

教学答案只抽取排名第一的有效 chunk，并把 source#section 绑定为引用。真实 RAG 可以放入多个 chunk，但必须受 token 预算和去重约束；无候选时应拒答或转人工，而不是凭语言模型记忆补写。

In [5]:
chunk_by_id = {chunk["id"]: chunk for chunk in chunks}  # 建立 chunk_id 到原始记录的索引
def assemble_answer(question, retrieval, character_budget=80):  # 定义带字符预算和引用的教学答案装配器
    if not retrieval["融合"]:  # 检查是否存在经过门禁的候选
        return {"answer": "知识库没有足够证据，转人工处理。", "citation": None, "chunk_id": None}  # 无证据时明确拒答
    top_id = retrieval["融合"][0]["chunk_id"]  # 选择融合排名第一的 chunk
    top_chunk = chunk_by_id[top_id]  # 读取候选正文与来源字段
    context = top_chunk["正文"][:character_budget]  # 按教学字符预算截取实际上下文
    citation = f"[{top_chunk['source']}#{top_chunk['section']}]"  # 从进入上下文的 chunk 生成引用
    return {"answer": context, "citation": citation, "chunk_id": top_id}  # 返回答案、引用和证据标识
answer_rows = []  # 创建逐问题结果表
for question in questions:  # 遍历六条用户问题
    assembled = assemble_answer(question, retrieval_runs[question["id"]])  # 装配当前问题的受控答案
    answer_rows.append({"问题": question["问题"], "期望chunk": question["expected"], "实际chunk": assembled["chunk_id"], "答案": assembled["answer"], "引用": assembled["citation"]})  # 保存可审计的逐样本结果
rag_hits = sum(row["期望chunk"] == row["实际chunk"] for row in answer_rows)  # 统计融合检索的 Top1 命中
print("逐问题答案与最终引用：")  # 输出答案结果标题
pprint(answer_rows)  # 展示每条答案实际使用的正文和引用
print(f"Top1 命中从 {baseline_hits}/{len(questions)} 提升到 {rag_hits}/{len(questions)}")  # 输出同数据下的检索变化

逐问题答案与最终引用：
[{'实际chunk': 'R_now',
  '引用': '[help/refund.md#到账时限]',
  '期望chunk': 'R_now',
  '答案': '退款审核通过后，通常三个工作日原路到账。',
  '问题': '退钱多久'},
 {'实际chunk': 'I_public',
  '引用': '[help/invoice.md#抬头修改]',
  '期望chunk': 'I_public',
  '答案': '在电子发票页修改公司名称和纳税号后重新开具。',
  '问题': '抬头和税号怎么改'},
 {'实际chunk': 'A_reset',
  '引用': '[help/account.md#密码重置]',
  '期望chunk': 'A_reset',
  '答案': '无法登录时先验证手机号，再重置密码。',
  '问题': '登录不了怎么处理'},
 {'实际chunk': 'O_cancel',
  '引用': '[help/order.md#取消订单]',
  '期望chunk': 'O_cancel',
  '答案': '未发货订单可在订单详情选择取消。',
  '问题': '未发货怎么撤单'},
 {'实际chunk': 'R_proof',
  '引用': '[help/refund.md#凭证要求]',
  '期望chunk': 'R_proof',
  '答案': '商品破损退款需要上传外包装和商品照片。',
  '问题': '破损退款要什么证明'},
 {'实际chunk': 'L_address',
  '引用': '[help/logistics.md#地址修改]',
  '期望chunk': 'L_address',
  '答案': '发货前可在订单详情修改收货地址。',
  '问题': '收货地还能改吗'}]
Top1 命中从 3/6 提升到 6/6


## 失败案例：在生成后才过滤 ACL

“先把所有候选交给模型，再从显示结果里删掉内部文档”已经太晚，因为内部正文可能进入 prompt、日志或缓存。下面让 Q2 在无门禁时召回字段最匹配的内部审批稿，再展示前置门禁后的公开引用。

In [6]:
unsafe_q2 = retrieve_question(questions[1], audience="public", enforce_gate=False)  # 错误地允许所有可见性和版本参与召回
unsafe_answer = assemble_answer(questions[1], unsafe_q2)  # 把未过滤候选直接装入答案上下文
safe_answer = assemble_answer(questions[1], retrieval_runs["Q2"])  # 使用前置门禁后的候选装配答案
print("失败案例：晚过滤前已经选中的内部证据")  # 输出失败行为标题
pprint(unsafe_answer)  # 展示内部稿进入上下文与引用的证据
print("修正：召回前执行 active + visibility 门禁")  # 输出修正行为标题
pprint(safe_answer)  # 展示只引用公开帮助文档的结果

失败案例：晚过滤前已经选中的内部证据
{'answer': '内部审批账号与操作口令仅供员工使用。',
 'chunk_id': 'I_internal',
 'citation': '[internal/invoice.md#审批]'}
修正：召回前执行 active + visibility 门禁
{'answer': '在电子发票页修改公司名称和纳税号后重新开具。',
 'chunk_id': 'I_public',
 'citation': '[help/invoice.md#抬头修改]'}


## 生产差距

线上还要处理真实分词、拼写纠错、字段 BM25、租户 ACL、文档生效时间、chunk 去重与 token 预算。候选和 prompt 应记录版本但避免写入敏感正文；引用还需做答案—证据蕴含校验，并监控无答案率、召回率、陈旧引用、延迟和缓存失效。

In [7]:
assert len(chunks) == 8  # 验证案例包含八个带审计字段的真实语义 chunk
assert baseline_hits < rag_hits  # 验证融合与门禁优于原词重叠基线
assert rag_hits == len(questions)  # 验证六条问题均选择人工期望证据
assert unsafe_answer["chunk_id"] == "I_internal"  # 验证晚过滤会让内部稿进入上下文
assert safe_answer["chunk_id"] == "I_public"  # 验证前置门禁恢复公开发票文档
assert all(row["引用"] is not None and row["实际chunk"] in chunk_by_id for row in answer_rows)  # 验证每条答案都绑定实际检索证据
print("最小回归测试通过：改写、融合、引用与 ACL 门禁均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：改写、融合、引用与 ACL 门禁均满足预期
